El servicio de venta de autos usados Rusty Bargain está desarrollando una aplicación para atraer nuevos clientes. Gracias a esa app, puedes averiguar rápidamente el valor de mercado de tu coche. Tienes acceso al historial: especificaciones técnicas, versiones de equipamiento y precios. Tienes que crear un modelo que determine el valor de mercado.
A Rusty Bargain le interesa:
- la calidad de la predicción;
- la velocidad de la predicción;
- el tiempo requerido para el entrenamiento

Significado de las columnas de los datos:

- DateCrawled — fecha en la que se descargó el perfil de la base de datos
- VehicleType — tipo de carrocería del vehículo
- RegistrationYear — año de matriculación del vehículo
- Gearbox — tipo de caja de cambios
- Power — potencia (CV)
- Model — modelo del vehículo
- Mileage — kilometraje (medido en km de acuerdo con las especificidades regionales del conjunto de datos)
- RegistrationMonth — mes de matriculación del vehículo
- FuelType — tipo de combustible
- Brand — marca del vehículo
- NotRepaired — vehículo con o sin reparación
- DateCreated — fecha de creación del perfil
- NumberOfPictures — número de fotos del vehículo
- PostalCode — código postal del propietario del perfil (usuario)
- LastSeen — fecha de la última vez que el usuario estuvo activo

### Imports

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import (OneHotEncoder, 
                                   StandardScaler)
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

import joblib


### Preparación de datos

#### Cargar datos

In [2]:
nombre_de_archivo = "car_data.csv"
ruta_a_datasets = "datasets"
ruta_completa = os.path.join(ruta_a_datasets, nombre_de_archivo)

In [3]:
df = pd.read_csv(ruta_completa)

#### Análisis exploratorio de datos (EDA)

In [4]:
# Ver la información del DataFrame
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 354369 entries, 0 to 354368
Data columns (total 16 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   DateCrawled        354369 non-null  object
 1   Price              354369 non-null  int64 
 2   VehicleType        316879 non-null  object
 3   RegistrationYear   354369 non-null  int64 
 4   Gearbox            334536 non-null  object
 5   Power              354369 non-null  int64 
 6   Model              334664 non-null  object
 7   Mileage            354369 non-null  int64 
 8   RegistrationMonth  354369 non-null  int64 
 9   FuelType           321474 non-null  object
 10  Brand              354369 non-null  object
 11  NotRepaired        283215 non-null  object
 12  DateCreated        354369 non-null  object
 13  NumberOfPictures   354369 non-null  int64 
 14  PostalCode         354369 non-null  int64 
 15  LastSeen           354369 non-null  object
dtypes: int64(7), object(

In [5]:
df.describe()

,Price,RegistrationYear,Power,Mileage,RegistrationMonth,NumberOfPictures,PostalCode
count,354369.000000,354369.000000,354369.000000,354369.000000,354369.000000,354369.0,354369.000000
mean,4416.656776,2004.234448,110.094337,128211.172535,5.714645,0.0,50508.689087
std,4514.158514,90.227958,189.850405,37905.341530,3.726421,0.0,25783.096248
min,0.000000,1000.000000,0.000000,5000.000000,0.000000,0.0,1067.000000
25%,1050.000000,1999.000000,69.000000,125000.000000,3.000000,0.0,30165.000000
50%,2700.000000,2003.000000,105.000000,150000.000000,6.000000,0.0,49413.000000
75%,6400.000000,2008.000000,143.000000,150000.000000,9.000000,0.0,71083.000000
max,20000.000000,9999.000000,20000.000000,150000.000000,12.000000,0.0,99998.000000


##### Price

In [6]:
# Se observan muchos autos con precios inferiores a 30 dólares. Se decide explorar un poco más
df[df["Price"] < 500] # Explorar modelos de autos con precio menor a 500 dólares

,DateCrawled,Price,VehicleType,RegistrationYear,Gearbox,Power,Model,Mileage,RegistrationMonth,FuelType,Brand,NotRepaired,DateCreated,NumberOfPictures,PostalCode,LastSeen
0,24/03/2016 11:52,480,NaN,1993,manual,0,golf,150000,0,petrol,volkswagen,NaN,24/03/2016 00:00,0,70435,07/04/2016 03:16
7,21/03/2016 18:54,0,sedan,1980,manual,50,other,40000,7,petrol,volkswagen,no,21/03/2016 00:00,0,19348,25/03/2016 16:47
15,11/03/2016 21:39,450,small,1910,NaN,0,ka,5000,0,petrol,ford,NaN,11/03/2016 00:00,0,24148,19/03/2016 08:46
16,01/04/2016 12:46,300,NaN,2016,NaN,60,polo,150000,0,petrol,volkswagen,NaN,01/04/2016 00:00,0,38871,01/04/2016 12:46
23,12/03/2016 19:43,450,small,1997,manual,50,arosa,150000,5,petrol,seat,no,12/03/2016 00:00,0,9526,21/03/2016 01:46
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
354318,15/03/2016 19:57,400,wagon,1991,manual,0,legacy,150000,0,petrol,subaru,NaN,15/03/2016 00:00,0,24558,19/03/2016 15:49
354329,30/03/2016 20:55,350,small,1996,NaN,65,punto,150000,0,NaN,fiat,NaN,30/03/2016 00:00,0,25436,07/04/2016 13:50
354335,04/04/2016 14:41,390,small,1997,auto,0,corsa,100000,6,petrol,opel,yes,04/04/2016 00:00,0,17509,06/04/2016 15:46
354338,31/03/2016 19:52,180,NaN,1995,NaN,0,NaN,125000,3,petrol,opel,NaN,31/03/2016 00:00,0,41470,06/04/2016 14:18


In [7]:
# Se aprecian autos cuyos precios no están acorde a la realidad; ejemplo el Polo de Volkswagen tiene un precio de 300 dólares; aunque tenga 150 mil millas;
# no tiene reparaciones; valorado en el mercado entre 2000 a 3000 dólares (búsqueda de internet). 
# Se decide eliminar autos con precio menor a 500 dólares; para mantener el modelo simple y no tener que hacer una limpieza más profunda de los datos.
df_filtrado = df[df["Price"] >= 500]

In [8]:
len(df_filtrado)

318315

##### VehicleType

In [9]:
# Explorar tipos de vehículos.
df_filtrado[df_filtrado["VehicleType"].isna()]

,DateCrawled,Price,VehicleType,RegistrationYear,Gearbox,Power,Model,Mileage,RegistrationMonth,FuelType,Brand,NotRepaired,DateCreated,NumberOfPictures,PostalCode,LastSeen
22,23/03/2016 14:52,2900,NaN,2018,manual,90,meriva,150000,5,petrol,opel,no,23/03/2016 00:00,0,49716,31/03/2016 01:16
26,10/03/2016 19:38,5555,NaN,2017,manual,125,c4,125000,4,NaN,citroen,no,10/03/2016 00:00,0,31139,16/03/2016 09:16
31,29/03/2016 16:57,899,NaN,2016,manual,60,clio,150000,6,petrol,renault,NaN,29/03/2016 00:00,0,37075,29/03/2016 17:43
37,28/03/2016 17:50,1500,NaN,2016,NaN,0,kangoo,150000,1,gasoline,renault,no,28/03/2016 00:00,0,46483,30/03/2016 09:18
48,25/03/2016 14:40,7750,NaN,2017,manual,80,golf,100000,1,petrol,volkswagen,NaN,25/03/2016 00:00,0,48499,31/03/2016 21:47
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
354336,05/03/2016 18:57,3299,NaN,2005,auto,0,outlander,150000,4,petrol,mitsubishi,NaN,05/03/2016 00:00,0,17034,06/03/2016 04:17
354346,07/03/2016 17:06,2600,NaN,2005,auto,0,c_klasse,150000,9,NaN,mercedes_benz,NaN,07/03/2016 00:00,0,61169,08/03/2016 21:28
354351,11/03/2016 23:40,1900,NaN,2000,manual,110,NaN,150000,7,NaN,volkswagen,no,11/03/2016 00:00,0,87700,12/03/2016 14:16
354361,09/03/2016 13:37,5250,NaN,2016,auto,150,159,150000,12,NaN,alfa_romeo,no,09/03/2016 00:00,0,51371,13/03/2016 01:44


In [10]:
# Se deciden eliminar todos los vehículos que se desconoce su tipo; porque no se puede inferir el tipo de vehículo y no se puede reemplazar por un valor genérico.
# Además, se sabe que el tipo de vehículo es importante para el precio. Por ejemplo; un SUV es más caro que un sedán.
df_filtrado = df_filtrado[df_filtrado["VehicleType"].notna()]

In [11]:
len(df_filtrado)

291221

##### Registration Year

In [12]:
# En la descripción se aprecia que el año de registro tiene un mínimo de 1000 y un máximo de 9999, lo cual no es realista. 
# Se decide eliminar autos cuyo registro sea inferior a 1990 y superior a 2026.
# Los precios de autos varian mucho; los autos viejos y de colección son autos muy caros; al igual que los autos nuevos.
# Para mantener el modelo simple, solo se dejaran autos "modernos" y se eliminarán los autos antíguos; porque se desconoce si son de colección o solo meten ruido al modelo.
df_filtrado = df_filtrado[(df_filtrado["RegistrationYear"] >= 1990) & (df_filtrado["RegistrationYear"] <= 2026)]
df_filtrado["RegistrationYear"].unique()

array([2011, 2004, 2001, 2008, 1995, 2014, 1998, 2005, 2007, 2009, 2002,
       1990, 2003, 1991, 2006, 1999, 2012, 2010, 2000, 1992, 1993, 2013,
       1994, 1997, 1996, 2015, 2016, 2017, 2018])

In [13]:
len(df_filtrado)

283508

##### Gearbox

In [14]:
# Explorar Gearbox
df_filtrado["Gearbox"].unique()

array(['manual', 'auto', nan], dtype=object)

In [15]:
# Ahora se reemplazaran los valores ausentes de la columna "Gearbox" por "manual". 
df_filtrado["Gearbox"] = df_filtrado["Gearbox"].fillna("manual") # Reemplazar valores ausentes por "manual"

In [16]:
len(df_filtrado)

283508

##### Power

In [17]:
# Así mismo se observa que hay autos que tiene una potencia (Power) de 0, esto tampoco es realista; se exploran los tipos de autos que tienen potencia menor a 30 
# para decidir si eliminar o reemplazar.
df_filtrado[(df_filtrado["Power"] <30)]["Model"].unique()

array(['signum', 'astra', 'polo', 'a4', 'combo', 'golf', nan, '3er', 'a3',
       '7er', 'other', 'c_klasse', 'corsa', 'sharan', '2_reihe', '5er',
       'touran', 'cooper', 'lupo', 'x_reihe', 'punto', 'mondeo', 'a2',
       'twingo', 'fabia', 's_klasse', 'focus', 'civic', '1er', 'caddy',
       'kalos', 'laguna', 'transit', 'omega', 'micra', 'clio', 'vito',
       'clk', 'a_klasse', 'ka', 'escort', 'fiesta', 'passat', 'modus',
       'a5', 'octavia', 'agila', 'antara', 'kangoo', 'touareg', 'picanto',
       'bora', 'q7', 'e_klasse', '3_reihe', 'fortwo', 'insignia',
       '1_reihe', 'leon', 'jetta', 'almera', 'tigra', '6_reihe', 'scenic',
       'verso', 'zafira', 'primera', 'meriva', 'i_reihe', 'mx_reihe',
       'roadster', '147', 'sandero', 'beetle', 'ibiza', 'a6', 'rav',
       'arosa', 'matiz', 'transporter', 'espace', 'c1', 'rio', 'grand',
       'niva', 'corolla', 'vectra', 'logan', '850', 'cuore', 'v40',
       'getz', 'fox', 'superb', 'seicento', 'ypsilon', 'megane',
       '

In [18]:
# Se decide eliminar estos registros; hay todo tipo de vehículos en esta lista, carros de golf, camionetas, autos de lujo. Mantener estos registros afectará al modelo.
# En internet se consiguió que los autos promedios tienen una potencia promedio de 100 CV (autos); sin embargo hay autos como el Bettle (Volkswagen) que tiene una potencia de 75 CV, por lo que se decide eliminar autos con potencia de 35 CV.
# Esto coincide con el promedio obtenido en la descripción del DataFrame.
# Se decide dejar solamente autos con potencia mayor o igual a 30 CV
df_filtrado = df_filtrado[df_filtrado["Power"] >= 30]

In [19]:
len(df_filtrado)

264549

##### Model

In [20]:
df_filtrado["Model"].unique()

array([nan, 'grand', 'golf', 'fabia', '3er', '2_reihe', 'c_max',
       '3_reihe', 'passat', 'navara', 'twingo', 'a_klasse', 'scirocco',
       '5er', 'other', 'civic', 'punto', 'e_klasse', 'kadett', 'one',
       'fortwo', 'clio', '1er', 'b_klasse', 'a8', 'jetta', 'fiesta',
       'c_klasse', 'micra', 'vito', 'sprinter', 'escort', 'forester',
       'xc_reihe', 'scenic', 'a1', 'transporter', 'focus', 'a4', 'tt',
       'astra', 'a6', 'jazz', 'polo', 'slk', '7er', 'combo', '80', '147',
       'z_reihe', 'sorento', 'ibiza', 'mustang', 'eos', 'touran', 'getz',
       'insignia', 'ka', 'megane', 'a3', 'lupo', 'mondeo', 'cordoba',
       'colt', 'impreza', 'corsa', 'vectra', 'berlingo', 'tiguan',
       '6_reihe', 'c4', 'panda', 'up', 'i_reihe', 'ceed', 'kangoo',
       '5_reihe', 'yeti', 'octavia', 'zafira', 'mii', 'rx_reihe', 'fox',
       'matiz', 'beetle', 'rio', 'touareg', 'logan', 'caddy', 'spider',
       'omega', 'cuore', 's_max', 'modus', 'a2', 'galaxy', 'c3', 'viano',
       's_k

In [21]:
df_filtrado[df_filtrado["Model"].isna()]

,DateCrawled,Price,VehicleType,RegistrationYear,Gearbox,Power,Model,Mileage,RegistrationMonth,FuelType,Brand,NotRepaired,DateCreated,NumberOfPictures,PostalCode,LastSeen
1,24/03/2016 10:58,18300,coupe,2011,manual,190,NaN,125000,5,gasoline,audi,yes,24/03/2016 00:00,0,66954,07/04/2016 01:46
135,27/03/2016 20:51,1450,sedan,1992,manual,136,NaN,150000,0,NaN,audi,no,27/03/2016 00:00,0,38709,05/04/2016 20:17
151,27/03/2016 20:47,6799,small,2009,manual,60,NaN,20000,5,petrol,volkswagen,no,27/03/2016 00:00,0,89077,27/03/2016 20:47
161,28/03/2016 10:50,1495,wagon,2001,manual,64,NaN,150000,9,gasoline,volkswagen,NaN,28/03/2016 00:00,0,99086,04/04/2016 11:45
186,16/03/2016 15:51,14000,sedan,2008,manual,235,NaN,150000,0,NaN,bmw,no,12/02/2016 00:00,0,95131,07/04/2016 14:56
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
353964,10/03/2016 15:48,1599,small,2002,manual,75,NaN,150000,0,petrol,ford,NaN,10/03/2016 00:00,0,31655,20/03/2016 00:16
354062,19/03/2016 11:25,6000,small,2008,manual,155,NaN,150000,5,petrol,sonstige_autos,no,19/03/2016 00:00,0,63069,07/04/2016 00:46
354234,19/03/2016 01:47,5000,sedan,2002,auto,170,NaN,150000,0,petrol,audi,no,18/03/2016 00:00,0,85221,06/04/2016 03:45
354245,07/03/2016 16:37,560,small,2001,auto,170,NaN,90000,0,petrol,fiat,yes,07/03/2016 00:00,0,55743,12/03/2016 21:45


In [22]:
# Se decide eliminar autos cuyo modelo sea desconocido. El modelo influye mucho en el precio
df_filtrado = df_filtrado[df_filtrado["Model"].notna()]

In [23]:
len(df_filtrado)

256919

##### Brand

In [24]:
df_filtrado["Brand"].unique()

array(['jeep', 'volkswagen', 'skoda', 'bmw', 'peugeot', 'ford', 'mazda',
       'nissan', 'renault', 'mercedes_benz', 'honda', 'fiat', 'opel',
       'mini', 'smart', 'audi', 'subaru', 'volvo', 'mitsubishi',
       'alfa_romeo', 'kia', 'seat', 'hyundai', 'lancia', 'citroen',
       'toyota', 'chevrolet', 'dacia', 'suzuki', 'daihatsu', 'chrysler',
       'jaguar', 'rover', 'porsche', 'saab', 'daewoo', 'land_rover',
       'lada', 'trabant'], dtype=object)

In [25]:
df_filtrado[df_filtrado["Brand"].isna()]

,DateCrawled,Price,VehicleType,RegistrationYear,Gearbox,Power,Model,Mileage,RegistrationMonth,FuelType,Brand,NotRepaired,DateCreated,NumberOfPictures,PostalCode,LastSeen


##### Fuel Type

In [26]:
# Explorar FuelType
df_filtrado["FuelType"].unique()

array(['gasoline', 'petrol', nan, 'lpg', 'other', 'hybrid', 'cng',
       'electric'], dtype=object)

In [27]:
# Explorar FuelType
print("Tipos de combustible en el registro de autos:")
print(df_filtrado[df_filtrado["FuelType"].isnull()]["VehicleType"].unique())
print("")
print("Años de registro únicos de autos con FuelType nulo:")
print(df_filtrado[df_filtrado["FuelType"].isnull()]["RegistrationYear"].unique())
print("")    
print("Cantidad de autos con FuelType nulo:")
print(len(df_filtrado[df_filtrado["FuelType"].isnull()]))

Tipos de combustible en el registro de autos:
['small' 'wagon' 'other' 'sedan' 'bus' 'convertible' 'coupe' 'suv']

Años de registro únicos de autos con FuelType nulo:
[1998 2004 1991 2002 2008 1996 1999 2000 2006 2001 2003 2009 2010 1995
 1997 2005 1993 2011 1992 1990 2007 1994 2012 2014 2013 2015 2017 2016]

Cantidad de autos con FuelType nulo:
7425


In [28]:
# Se decide eliminar estos registros; ya que es posible que hayan autos híbridos o eléctricos dentro del registro.
df_filtrado = df_filtrado[df_filtrado["FuelType"].notna()] # El tipo de combustible afecta un poco el precio, pero son pocos registros

In [29]:
len(df_filtrado)

249494

##### Repairs

In [30]:
df_filtrado[df_filtrado["NotRepaired"].isna()]

,DateCrawled,Price,VehicleType,RegistrationYear,Gearbox,Power,Model,Mileage,RegistrationMonth,FuelType,Brand,NotRepaired,DateCreated,NumberOfPictures,PostalCode,LastSeen
2,14/03/2016 12:52,9800,suv,2004,auto,163,grand,125000,8,gasoline,jeep,NaN,14/03/2016 00:00,0,90480,05/04/2016 12:47
8,04/04/2016 23:42,14500,bus,2014,manual,125,c_max,30000,8,petrol,ford,NaN,04/04/2016 00:00,0,94505,04/04/2016 23:42
12,15/03/2016 22:49,999,wagon,1995,manual,115,passat,150000,11,petrol,volkswagen,NaN,15/03/2016 00:00,0,37269,01/04/2016 13:16
42,24/03/2016 00:52,12500,sedan,2006,auto,231,5er,150000,11,gasoline,bmw,NaN,23/03/2016 00:00,0,46119,04/04/2016 16:18
44,17/03/2016 12:44,3900,small,2008,auto,61,fortwo,80000,6,petrol,smart,NaN,17/03/2016 00:00,0,21073,19/03/2016 11:46
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
354341,11/03/2016 15:50,699,sedan,1999,manual,101,vectra,150000,3,petrol,opel,NaN,11/03/2016 00:00,0,65936,17/03/2016 12:44
354349,16/03/2016 17:06,5999,wagon,2005,manual,140,a4,150000,4,gasoline,audi,NaN,16/03/2016 00:00,0,56472,18/03/2016 11:30
354356,04/04/2016 11:45,999,convertible,2000,manual,95,megane,150000,4,petrol,renault,NaN,04/04/2016 00:00,0,88477,06/04/2016 12:44
354357,09/03/2016 11:36,1690,wagon,2004,manual,55,fabia,150000,4,petrol,skoda,NaN,09/03/2016 00:00,0,18246,04/04/2016 08:15


In [31]:
df_filtrado["NotRepaired"] = df_filtrado["NotRepaired"].fillna("no") # Reemplazar valores ausentes por "No"
df_filtrado["NotRepaired"].unique()

array(['no', 'yes'], dtype=object)

In [32]:
len(df_filtrado)

249494

In [33]:
# Ahora con los datos filtrados; se decide dejar unicamente las columnas que aporten información útil para el modelo:
# Price; VehicleType; RegistrationYear; Gearbox; Power; Mileage; FuelType; Brand; NotRepaired

In [34]:
df_filtrado.info()

<class 'pandas.core.frame.DataFrame'>
Index: 249494 entries, 2 to 354368
Data columns (total 16 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   DateCrawled        249494 non-null  object
 1   Price              249494 non-null  int64 
 2   VehicleType        249494 non-null  object
 3   RegistrationYear   249494 non-null  int64 
 4   Gearbox            249494 non-null  object
 5   Power              249494 non-null  int64 
 6   Model              249494 non-null  object
 7   Mileage            249494 non-null  int64 
 8   RegistrationMonth  249494 non-null  int64 
 9   FuelType           249494 non-null  object
 10  Brand              249494 non-null  object
 11  NotRepaired        249494 non-null  object
 12  DateCreated        249494 non-null  object
 13  NumberOfPictures   249494 non-null  int64 
 14  PostalCode         249494 non-null  int64 
 15  LastSeen           249494 non-null  object
dtypes: int64(7), object(9)
me

In [35]:
# Nota; aunque es posible que el código postal pueda influir en el precio del auto; se decide eliminar esta columna como entrada del modelo.
# Lo ideal sería segmentar autos con mismas características para cada código postal; luego habría que hacer una t de student para evaluar si existe
# diferencia estadísticamente significativa entre la media de los precios. 
# Otra forma sería hacer un análisis bootstrapping para estimar la media y la desviación estándar de los precios para cada código postal; 
# luego comparar los intervalos de confianza de cada código postal. De hecho, esto sería lo mejor porque no se asume que los precios 
# de los autos tengan una distribución normal. Y además, incorporla la variabilidad de todos los precios de autos sin centrarse en un tipo
# en específico.
 
data = df_filtrado[["Price", "VehicleType", "RegistrationYear", "Gearbox", "Power", "Mileage", "FuelType", "Brand", "NotRepaired"]]

In [36]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 249494 entries, 2 to 354368
Data columns (total 9 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   Price             249494 non-null  int64 
 1   VehicleType       249494 non-null  object
 2   RegistrationYear  249494 non-null  int64 
 3   Gearbox           249494 non-null  object
 4   Power             249494 non-null  int64 
 5   Mileage           249494 non-null  int64 
 6   FuelType          249494 non-null  object
 7   Brand             249494 non-null  object
 8   NotRepaired       249494 non-null  object
dtypes: int64(4), object(5)
memory usage: 19.0+ MB


##### Conclusion del EDA y limpieza de datos

- Se filtraron los datos COMPLETAR

## Entrenamiento del modelo 

### Preprocesado

In [37]:
# Explicación del procedimiento:
# 1. Dividir los datos en target y features de la siguiente forma:
# Target = Price
# Features = VehicleType, RegistrationYear, Gearbox, Power, Mileage, FuelType, Brand, NotRepaired

# 2. Dividir los datos en datos de prueba y de entrenamiento en proporción 75% entrenamiento y 25% prueba 
# NOTA: usaré train_test_split en lugar de cross_val_score porque voy a entrenar muchos modelos y el poder de computo es limitado.

# 3. Crear dos sets de datos:
# 3.1. Datos SIN encoding y estandarizados para entrenar el modelo CatBoost y LightGBM
# 3.2. Datos CON encoding y estandarizados para entrenar todos los demás modelos.


In [38]:
# 1. Datos sin encoding para entrenar el modelo CatBoost y LightGBM
features = data.drop("Price", axis=1)
target = data["Price"]

In [39]:
# 2. Dividir los datos en datos de prueba y de entrenamiento en proporción 75% entrenamiento y 25% prueba
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.25, random_state=12345)

In [40]:
# 3. Crear dos sets de datos:

# 3.1. Datos SIN encoding y estandarizados para entrenar el modelo CatBoost y LightGBM
X_train_no_encoding = X_train.copy()
X_test_no_encoding = X_test.copy()

In [41]:
#3.2# Separar variables categóricas y numéricas
cat_cols = ["VehicleType",
            "Gearbox",
            "FuelType",
            "Brand",
            "NotRepaired"]

num_cols = ["RegistrationYear",
            "Power",
            "Mileage"]

# Crear el preprocesador
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore') 
scaler = StandardScaler()

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", ohe, cat_cols),
        ("num", scaler, num_cols)
    ]
)

X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

In [42]:
# Los resultados de correr cada modelo se guardarán en el siguiente diccionario:
results = {
    "datasets": {
        "X_train_no_encoding": X_train_no_encoding,
        "X_test_no_encoding": X_test_no_encoding,
        "X_train_encoded": X_train_encoded,
        "X_test_encoded": X_test_encoded,
        "y_train": y_train,
        "y_test": y_test
    },

    "Lineal Regression": {
        "r2_score": None,
        "rmse": None,
        "mae": None,
        "y_pred": None,
        "model": None
    },
    "Decision Tree": {
        "r2_score": None,
        "rmse": None,
        "mae": None,
        "best_depth": None,
        "best_leaf_size": None,
        "y_pred": None,
        "model": None
    },
    "Random Forest": {
        "r2_score": None,
        "rmse": None,
        "mae": None,
        "best_depth": None,
        "best_leaf_size": None,
        "best_estimators": None,
        "y_pred": None,
        "model": None
    },
    "XGBoost": {
        "r2_score": None,
        "rmse": None,
        "mae": None,
        "y_pred": None,
        "model": None
    },
    "LightGBM": {
        "r2_score": None,
        "rmse": None,
        "mae": None,
        "y_pred": None,
        "model": None
    },
    "CatBoost": {
        "r2_score": None,
        "rmse": None,
        "mae": None,
        "y_pred": None,
        "model": None
    }
}

### Regresion lineal

In [ ]:
# model = LinearRegression()
# model.fit(X_train_encoded, y_train)
# y_pred = model.predict(X_test_encoded)
# r2 = r2_score(y_test, y_pred)
# rmse = mean_squared_error(y_test, y_pred)**0.5
# mae = mean_absolute_error(y_test, y_pred)

# print("******************************** LINEAR REGRESSION ********************************")
# print("R2 score:", r2)
# print("Root Mean Squared Error:", rmse)
# print("Mean Absolute Error:", mae)
# print("")

# results["Lineal Regression"]["r2_score"] = r2
# results["Lineal Regression"]["rmse"] = rmse
# results["Lineal Regression"]["mae"] = mae
# results["Lineal Regression"]["y_pred"] = y_pred
# results["Lineal Regression"]["model"] = model


******************************** LINEAR REGRESSION ********************************
R2 score: 0.6969011951585037
Root Mean Squared Error: 2539.137696633106
Mean Absolute Error: 1861.3909308364073



###  Árbol de decisión 

In [ ]:
# best_depth = 0
# best_leaf_size = 0
# best_r2 = float('-inf')  # Initialize best R2 score to negative infinity
# best_rmse = float('inf')  # Initialize best RMSE to positive infinity

# depths = [5, 7, 10, 15]
# leaf_sizes = [5, 10, 20, 50]
# for depth in depths:
#     for leaf_size in leaf_sizes:
#         model = DecisionTreeRegressor(max_depth=depth, min_samples_leaf=leaf_size, random_state=12345)
#         model.fit(X_train_encoded, y_train)
#         y_pred = model.predict(X_test_encoded)
#         r2 = r2_score(y_test, y_pred)
#         rmse = mean_squared_error(y_test, y_pred)**0.5
#         mae = mean_absolute_error(y_test, y_pred)
#         print(f"Depth: {depth}, Leaf Size: {leaf_size}")
#         print("R2 score:", r2)
#         print("Root Mean Squared Error:", rmse)
#         print("Mean Absolute Error:", mae)
#         print("-----------------------------")
#         if r2 > best_r2:
#             best_r2 = r2
#             best_rmse = rmse
#             best_mae = mae
#             best_depth = depth
#             best_leaf_size = leaf_size
#             y_pred_best = y_pred
#             best_model = model

# print("******************************** DECISION TREE BEST MODEL ********************************")
# print("Best Depth:", best_depth)
# print("Best Leaf Size:", best_leaf_size)
# print("Best R2 Score:", best_r2)
# print("Best RMSE:", best_rmse)
# print("Best MAE:", best_mae)
# print("")

# results["Decision Tree"]["r2_score"] = best_r2
# results["Decision Tree"]["rmse"] = best_rmse
# results["Decision Tree"]["mae"] = best_mae
# results["Decision Tree"]["best_depth"] = best_depth
# results["Decision Tree"]["best_leaf_size"] = best_leaf_size
# results["Decision Tree"]["y_pred"] = y_pred_best
# results["Decision Tree"]["model"] = best_model

Depth: 5, Leaf Size: 5
R2 score: 0.7557339210437797
Root Mean Squared Error: 2279.4271758558793
Mean Absolute Error: 1574.679149044919
-----------------------------
Depth: 5, Leaf Size: 10
R2 score: 0.7557339210437797
Root Mean Squared Error: 2279.4271758558793
Mean Absolute Error: 1574.679149044919
-----------------------------
Depth: 5, Leaf Size: 20
R2 score: 0.7557339210437797
Root Mean Squared Error: 2279.4271758558793
Mean Absolute Error: 1574.679149044919
-----------------------------
Depth: 5, Leaf Size: 50
R2 score: 0.7557339210437797
Root Mean Squared Error: 2279.4271758558793
Mean Absolute Error: 1574.679149044919
-----------------------------
Depth: 7, Leaf Size: 5
R2 score: 0.7967003734944923
Root Mean Squared Error: 2079.516791910104
Mean Absolute Error: 1423.5848941375891
-----------------------------
Depth: 7, Leaf Size: 10
R2 score: 0.7967400041303965
Root Mean Squared Error: 2079.314094563922
Mean Absolute Error: 1423.2206077898793
-----------------------------
Depth:

### Bosques aleatorios (Random Forest)

In [ ]:
# depths = [5, 7, 10, 15]
# leaf_sizes = [5, 10, 20, 50]
# n_estimators = [50, 100, 200]

# best_r2 = float('-inf')  # Initialize best R2 score to negative infinity
# best_rmse = float('inf')  # Initialize best RMSE to positive infinity


# for depth in depths:
#     for leaf_size in leaf_sizes:
#         for n_estimator in n_estimators:
#             model = RandomForestRegressor(n_estimators=n_estimator, max_depth=depth, min_samples_leaf=leaf_size, random_state=12345)
#             model.fit(X_train_encoded, y_train)
#             y_pred = model.predict(X_test_encoded)
#             r2 = r2_score(y_test, y_pred)
#             rmse = mean_squared_error(y_test, y_pred)**0.5
#             mae = mean_absolute_error(y_test, y_pred)
#             print(f"Depth: {depth}, Leaf Size: {leaf_size}, N Estimators: {n_estimator}")
#             print("R2 score:", r2)
#             print("Root Mean Squared Error:", rmse)
#             print("Mean Absolute Error:", mae)
#             print("-----------------------------")
#             if r2 > best_r2:
#                 best_r2 = r2
#                 best_rmse = rmse
#                 best_mae = mae
#                 best_depth = depth
#                 best_leaf_size = leaf_size
#                 best_n_estimators = n_estimator
#                 y_pred_best = y_pred
#                 best_model = model

# print("******************************** RANDOM FOREST BEST MODEL ********************************")
# print("Best Depth:", best_depth)
# print("Best Leaf Size:", best_leaf_size)
# print("Best N Estimators:", best_n_estimators)
# print("Best R2 Score:", best_r2)
# print("Best RMSE:", best_rmse)
# print("Best MAE:", best_mae)
# print("")

# results["Random Forest"]["r2_score"] = best_r2
# results["Random Forest"]["rmse"] = best_rmse
# results["Random Forest"]["mae"] = best_mae
# results["Random Forest"]["best_depth"] = best_depth
# results["Random Forest"]["best_leaf_size"] = best_leaf_size
# results["Random Forest"]["best_n_estimators"] = best_n_estimators
# results["Random Forest"]["y_pred"] = y_pred_best
# results["Random Forest"]["model"] = best_model


Depth: 5, Leaf Size: 5, N Estimators: 50
R2 score: 0.762202032159167
Root Mean Squared Error: 2249.045339637455
Mean Absolute Error: 1544.7487594407683
-----------------------------
Depth: 5, Leaf Size: 5, N Estimators: 100
R2 score: 0.7623243036621848
Root Mean Squared Error: 2248.4670556535725
Mean Absolute Error: 1544.6187012159023
-----------------------------
Depth: 5, Leaf Size: 5, N Estimators: 200
R2 score: 0.7622971603557847
Root Mean Squared Error: 2248.5954429661674
Mean Absolute Error: 1545.0705875326419
-----------------------------
Depth: 5, Leaf Size: 10, N Estimators: 50
R2 score: 0.762202032159167
Root Mean Squared Error: 2249.045339637455
Mean Absolute Error: 1544.7487594407683
-----------------------------
Depth: 5, Leaf Size: 10, N Estimators: 100
R2 score: 0.7623243036621848
Root Mean Squared Error: 2248.4670556535725
Mean Absolute Error: 1544.6187012159023
-----------------------------
Depth: 5, Leaf Size: 10, N Estimators: 200
R2 score: 0.7622971603557847
Root Me

### Guardar resultados de los 3 modelos en un archivo joblib

In [ ]:
# joblib.dump(results, "linear_tree_forest_models_results.joblib")

['linear_tree_forest_models_results.joblib']

In [ ]:
### Cargar resultados previos
results = joblib.load("linear_tree_forest_models_results.joblib")

### XGBoost

## Análisis del modelo

# Lista de control

Escribe 'x' para verificar. Luego presiona Shift+Enter

- [x]  Jupyter Notebook está abierto
- [ ]  El código no tiene errores- [ ]  Las celdas con el código han sido colocadas en orden de ejecución- [ ]  Los datos han sido descargados y preparados- [ ]  Los modelos han sido entrenados
- [ ]  Se realizó el análisis de velocidad y calidad de los modelos